# Проект 2: Анализ мутаций в прокариотическом геноме

- .SAM (sample alligned map?)- индексирование референсного генома
- .VCF (variant calling format) - формат для хранения SNP (single nucleotide polymorfism), которые были определены по референсу после выравнивания
- IGV - программа для визуализации покрытия генома, SNP и т.п.

Изучить механизм эволюции бактерии (E.Coli) в устойчивость к антибиотикам (намек на GEO и KEGG парсинг)


# Подготовительный этап

In [ ]:
mkdir -p TASK2 && cd TASK2 # Рабочая директория

conda activate biolab # Загрузка окружения

# Подготовка инструментов

conda install -c bioconda vcftools # Для оценки мутаций

conda create -n IGV -c bioconda igv # Подготовка окружения для визуализатора мутаций

# Содержание окружений

In [6]:
conda activate biolab && conda list

# packages in environment at /home/suunar/miniforge3/envs/biolab:
#
# Name                                   Version          Build                 Channel
_openmp_mutex                            4.5              20_gnu                conda-forge
about-time                               4.2.1            pyhd8ed1ab_1          https://mirrors.tuna.tsinghua.edu.cn/anaconda/cloud/conda-forge
abricate                                 1.2.0            h05cac1d_0            bioconda
alive-progress                           3.3.0            pypi_0                pypi
alsa-lib                                 1.2.3.2          h166bdaf_0            conda-forge
anndata                                  0.10.9           pypi_0                pypi
any2fasta                                0.8.1            hdfd78af_0            bioconda
aragorn                                  1.2.41           h7b50bb2_5            bioconda
archspec                                 0.2.5            pyhd8ed1ab_0         

In [7]:
conda activate IGV && conda list

# packages in environment at /home/suunar/miniforge3/envs/IGV:
#
# Name                       Version          Build             Channel
_openmp_mutex                4.5              20_gnu            conda-forge
alsa-lib                     1.2.15.3         hb03c661_0        conda-forge
bzip2                        1.0.8            hda65f42_9        conda-forge
ca-certificates              2026.2.25        hbd8a1cb_0        conda-forge
cairo                        1.18.4           he90730b_1        conda-forge
font-ttf-dejavu-sans-mono    2.37             hab24e00_0        conda-forge
font-ttf-inconsolata         3.000            h77eed37_0        conda-forge
font-ttf-source-code-pro     2.038            h77eed37_0        conda-forge
font-ttf-ubuntu              0.83             h77eed37_3        conda-forge
fontconfig                   2.17.1           h27c8c51_0        conda-forge
fonts-conda-ecosystem        1                0                 conda-forge
fonts-conda-forge          

In [8]:
conda activate SPAdes && conda list

# packages in environment at /home/suunar/miniforge3/envs/SPAdes:
#
# Name                                   Version          Build                 Channel
_openmp_mutex                            4.5              20_gnu                https://mirrors.tuna.tsinghua.edu.cn/anaconda/cloud/conda-forge
alsa-lib                                 1.2.15.3         hb03c661_0            https://mirrors.tuna.tsinghua.edu.cn/anaconda/cloud/conda-forge
aragorn                                  1.2.41           h7b50bb2_5            bioconda
argtable2                                2.13             hd590300_1004         conda-forge
barrnap                                  0.9              hdfd78af_4            bioconda
bedtools                                 2.31.1           hf5e1c6e_2            bioconda
blast                                    2.15.0           pl5321h6f7f691_1      bioconda
blast-legacy                             2.2.26           hf7ff83a_5            bioconda
bzip2              

# Первичная обработка ридов

In [ ]:
conda activate biolab

mkdir -p RawData && cd RawData # Папка для ридов и референсных данных

# Загрузка референсного генома и референсной аннотации

wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.gff.gz \
&& wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_cds_from_genomic.fna.gz \
&& wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/005/845/GCF_000005845.2_ASM584v2/GCF_000005845.2_ASM584v2_genomic.fna.gz

gunzip GCF_000005845.2_ASM584v2_cds_from_genomic.fna.gz && gunzip GCF_000005845.2_ASM584v2_genomic.gff.gz && gunzip GCF_000005845.2_ASM584v2_genomic.fna.gz

# Загрузка ридов через интерфейс гугл по общему пути /home/suunar/Downloads

# Перенос ридов в рабочую директорию

cd ~ && mv ./Downloads/amp_res_1.fastq.zip ./projects/Helloworld/TASK2/RawData \
&& mv mv ./Downloads/amp_res_2.fastq.zip ./projects/Helloworld/TASK2/RawData \
&& cd projects/Helloworld/TASK2/RawData

# Распаковка ридов

unzip amp_res_1.fastq.zip && unzip amp_res_2.fastq.zip

# Удаляем уже ненужные архивы ридов

rm amp_res_1.fastq.zip && rm amp_res_2.fastq.zip

# Контроль качества ридов fastqc

cd .. && mkdir -p FASTQCrep

cd FASTQCrep && mkdir -p Forw && mkdir -p Backw

fastqc -o Forw ../RawData/amp_res_1.fastq && fastqc -o Backw ../RawData/amp_res_2.fastq

# В целом, все не плохо, но нужно слегка подрезать концы ридов

cd .. && mkdir -p Trimmed

cd Trimmed && mkdir -p Paired && mkdir -p Unpaired && cd ..

trimmomatic PE ./RawData/amp_res_1.fastq ./RawData/amp_res_2.fastq \
            ./Trimmed/Paired/amp_res_1.fastq ./Trimmed/Unpaired/amp_res_1.fastq \
            ./Trimmed/Paired/amp_res_2.fastq ./Trimmed/Unpaired/amp_res_2.fastq \
            LEADING:20 TRAILING:20 SLIDINGWINDOW:10:20 MINLEN:20

# Получаем парные риды по пути ./Trimmed/Paired и непарные риды по пути ./Trimmed/Unpaired

# Теперь мы готовы к выравниванию на референс

# Маппинг

In [ ]:
# Индексирование

mkdir -p Index

bwa index ./RawData/GCF_000005845.2_ASM584v2_cds_from_genomic.fna

# Перенос файлов в созданную директорию вручную, потому что bwa дегенерат

mv ./RawData/GCF_000005845.2_ASM584v2_genomic.fna.amb ./Index && \
mv ./RawData/GCF_000005845.2_ASM584v2_genomic.fna.ann ./Index && \
mv ./RawData/GCF_000005845.2_ASM584v2_genomic.fna.bwt ./Index && \
mv ./RawData/GCF_000005845.2_ASM584v2_genomic.fna.pac ./Index && \
mv ./RawData/GCF_000005845.2_ASM584v2_genomic.fna.sa ./Index 

# Выравнивание

bwa mem -t 6 -M Index/GCF_000005845.2_ASM584v2_genomic.fna ./Trimmed/Paired/amp_res_1.fastq ./Trimmed/Paired/amp_res_2.fastq > aligned.sam

mkdir -p SAM && mv aligned.sam SAM # Файл выравнивания в отдельную папку для красоты

samtools view -S -b aligned.sam > aligned.bam # Бинаризация sam-файла в bam-файл

samtools flagstat aligned.bam # Первичная статистика по выравниванию (вывод ниже)

samtools sort aligned.bam -o aligned_sorted.bam # Сортировка

samtools index aligned_sorted.bam # Повторное индексирование

# В папке SAM должен создаться файл aligned_sorted.bam.bai, который будет использоваться в ближайшем будущем

```text
892776 + 0 in total (QC-passed reads + QC-failed reads)
892518 + 0 primary
258 + 0 secondary
0 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
891649 + 0 mapped (99.87% : N/A)
891391 + 0 primary mapped (99.87% : N/A)
892518 + 0 paired in sequencing
446259 + 0 read1
446259 + 0 read2
888566 + 0 properly paired (99.56% : N/A)
890412 + 0 with itself and mate mapped
979 + 0 singletons (0.11% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)

# Оценка мутаций и визуализация

In [ ]:
# Просмотр результатов выравнивания через интерфейс 

# Программа работает только с впн из-за переброса трафика через google((

conda activate IGV

igv

# После небольшого обзора выходим из программы на CTRL + C

conda activate biolab

conda install -c bioconda varscan

# Фиксируем количество снипов

samtools mpileup -f ./Index/GCF_000005845.2_ASM584v2_genomic.fna ./SAM/aligned_sorted.bam >  my.mpileup

# Получаем результаты в VCF файл

# В качестве фильтра пробуем значения 70%, 80%, 90% и 95% для необходимого кол-ва ридов для подтверждения мутации

mkdir -p VCF && cd VCF 

varscan mpileup2snp ../my.mpileup --min-var-freq 0.7 --variants --output-vcf 1 > VarScan_70_results.vcf # Зарегистрировано 6 снипов и 3 indel-а

varscan mpileup2snp ../my.mpileup --min-var-freq 0.8 --variants --output-vcf 1 > VarScan_80_results.vcf # Зарегистрировано 6 снипов и 3 indel-а

varscan mpileup2snp ../my.mpileup --min-var-freq 0.9 --variants --output-vcf 1 > VarScan_90_results.vcf # Зарегистрировано 6 снипов и 3 indel-а

varscan mpileup2snp ../my.mpileup --min-var-freq 0.95 --variants --output-vcf 1 > VarScan_95_results.vcf # Зарегистрировано 6 снипов и 0 indel-ов